<a href="https://colab.research.google.com/github/brpetros/prompts_and_evalution_notebooks/blob/main/2_prompt_execution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from pprint import pprint

# import the 70 selected skills
df_relevant_skills = pd.read_csv("/content/skill_sample.csv")
skills_sample = df_relevant_skills.to_dict(orient='records')


In [ ]:
import requests
from pprint import pprint
from google.colab import userdata
import os

BASE_URL = "https://skillab-tracker.csd.auth.gr/api"

def login(username, password):
    url = f"{BASE_URL}/login"

    # verify is set to false because a certification expired error was returned
    # this has to be changed to ensure a safe connection
    response = requests.post(url, json={"username": username, "password": password})
    response.raise_for_status()
    return response.json()

os.environ['API_TOKEN']= login(userdata.get("SKILLAB_USERNAME"), userdata.get("SKILLAB_PASSWORD"))

def auth_headers(token):
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type" : "application/x-www-form-urlencoded",
        "Accept": "application/json"
    }

# function to break an array into pieces
def chunked(iterable, size):
    for i in range(0, len(iterable), size):
        yield iterable[i:i + size]

# get skills data from IDs
def get_skills(token, skill_ids):
    url = f"{BASE_URL}/skills"
    CHUNK_SIZE = 25
    skills = {}

    for occ_chunk in chunked(skill_ids, CHUNK_SIZE):
        page = 1
        page_size = 100
        CHUNK_SIZE = 25
        MAX_PAGES = 1000

        while page <= MAX_PAGES:
            params = {"page": page, "page_size": page_size}
            payload = {"ids": skill_ids}

            response = requests.post(
                url,
                params=params,
                data=payload,
                headers=auth_headers(token),
            )
            response.raise_for_status()
            data = response.json()

            items = data.get("items", [])
            if not items:
                break

            for skill in items:
                skills[skill["id"]] = skill

            page += 1

    return {
        "count": len(skills),
        "skills": list(skills.values())
    }


def get_all_ancestors_with_labels(skills_data):
    """
      Takes the all the data from the selected skills.
      Extracts a dictionnary which contains all the ancestors with their labels.
    """
    all_ancestors = {}
    for skill in skills_sample_data["skills"]:
        ancestors = skill["skill_ancestors"] + skill["knowledge_ancestors"] + skill["traversal_ancestors"]
        for ancestor_group in ancestors:
            for ancestor in ancestor_group:
                all_ancestors[ancestor] = " "

    ancestors_ids = list(all_ancestors.keys())
    ancestors_data = get_skills(os.environ['API_TOKEN'], ancestors_ids)
    for ancestor in ancestors_data["skills"]:
        all_ancestors[ancestor["id"]] = ancestor["label"]
    return all_ancestors


def translate_skills_ancestors(skill, all_ancestors):
    skill_ancestors = skill["skill_ancestors"] + skill["knowledge_ancestors"] + skill["traversal_ancestors"]
    translated_ancestors = [ [all_ancestors[ancestor] for ancestor in ancestor_group] for ancestor_group in skill_ancestors]
    return translated_ancestors


## Creating a dictionary to store the labels for each ancestor

In [ ]:
#storing all the data regarding the skills we collected
skills_ids = [skill["skill_id"] for skill in skills_sample]
skills_sample_data = get_skills(os.environ['API_TOKEN'], skills_ids)

#creating a dictionary to easily access the label of each ancestor
all_ancestors = get_all_ancestors_with_labels(skills_sample_data)


### create the final input structure for each prompt
For each skill, we provide:
- the skill name
- the official esco description
- all the ancestors of the skill (skill, knowledge and traversal ancestors)


In [ ]:
input_data = {}

for skill in skills_sample_data['skills']:
    input_data[skill["id"]] = { "skill_name": skill["label"], "skill_description": skill["description"], "skill_ancestors": translate_skills_ancestors(skill, all_ancestors)}
input_data = list(input_data.values())
pprint(input_data)

In [ ]:
df = pd.DataFrame(input_data)
df.to_csv("input_data.csv")

In [ ]:

!pip install mistralai openai

## Interaction with LLMs API and record creation

In [ ]:
import os
from google import genai
import time
from mistralai.client import Mistral
from openai import OpenAI
from google.genai import types
from google.colab import userdata

# setting up the clients for all of the models
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

mistral = Mistral(api_key=userdata.get("MISTRAL_API_KEY"))

qwen_client = OpenAI(
        api_key=userdata.get("QWEN_API_KEY"),
        base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
    )


def create_record(latency,model,temperature,system_text,user_text,response_text,finish_reason,prompt_tokens=0,completion_tokens=0,total_tokens=0):
    """
      Creates the metadata record to be saved after each prompt.
      Contains an evaluation section to facilitate the next step.
    """
    return {
                "timestamp": datetime.now().isoformat(),
                "latency_seconds": round(latency, 2),
                "model": model ,
                "parameters": {
                    "temperature": temperature,
                    "system_instruction": system_text
                },
                "input": {
                    "prompt": user_text
                },
                "output": {
                    "text": response_text,
                    "finish_reason": finish_reason
                },
                "usage": {
                    "prompt_tokens": prompt_tokens,
                    "completion_tokens": completion_tokens,
                    "total_tokens": total_tokens
                },
                "evaluation": {
                    "clarity": 0,
                    "depth": 0,
                    "relevance": 0,
                    "pedagogical_value": 0,
                }
            }

def generate_gemini_response(user_text, system_text, temperature=0.3, model="gemini-3.1-flash-lite-preview"):
    try:
        contents = [
            types.Content(
                role="user",
                parts=[
                    types.Part.from_text(text=user_text),
                ],
            ),
        ]

        config = types.GenerateContentConfig(
            temperature=temperature,
            system_instruction=[
                types.Part.from_text(text=system_text)
            ],
            thinking_config=types.ThinkingConfig(thinking_budget=0),
            response_mime_type="application/json"
        )

        # generate response and calculate latency
        start_time = time.time()
        response = client.models.generate_content(
            model=model,
            contents=contents,
            config=config
        )
        latency = time.time() - start_time

        return create_record(latency=round(latency, 2), model=model, temperature=temperature, system_text=system_text, user_text=user_text, response_text=response.text, finish_reason=response.candidates[0].finish_reason.name, prompt_tokens=response.usage_metadata.prompt_token_count, completion_tokens=response.usage_metadata.candidates_token_count, total_tokens=response.usage_metadata.total_token_count)

    except Exception as e:
        print(f"Error message: {e}")


def generate_mistral_response(user_text, system_text, temperature=0.3, model="mistral-medium-latest"):
    try:
        start_time = time.time()
        response = mistral.chat.complete(
            model=model,
            messages=[{"role": "system","content": system_text}, {"role": "user","content": user_text}],
            temperature=temperature,
            stream=False,
            response_format={"type": "json_object"}
            )
        latency = time.time() - start_time

        return create_record(latency=round(latency, 2), model=response.model, temperature=temperature, system_text=system_text, user_text=user_text, response_text=response.choices[0].message.content, finish_reason=response.choices[0].finish_reason, prompt_tokens=response.usage.prompt_tokens, completion_tokens=response.usage.completion_tokens, total_tokens=response.usage.total_tokens)

    except Exception as e:
        print(f"Error message: {e}")

# for qwen we decided to use the qwen 3.5 open-source version
def generate_qwen_response(user_text, system_text, temperature=0.3, model="qwen3.5-35b-a3b"):
    try:
        start_time = time.time()
        completion = qwen_client.chat.completions.create(
            model=model,
            messages=[
                {'role': 'system', 'content': system_text},
                {'role': 'user', 'content': user_text}
            ],
            temperature=temperature,
            response_format={"type": "json_object"}
        )
        latency = time.time() - start_time
        return create_record(latency=round(latency, 2), model=completion.model, temperature=temperature, system_text=system_text, user_text=user_text, response_text=completion.choices[0].message.content, finish_reason=completion.choices[0].finish_reason, prompt_tokens=completion.usage.prompt_tokens, completion_tokens=completion.usage.completion_tokens, total_tokens=completion.usage.total_tokens)
    except Exception as e:
        print(f"Error message: {e}")
        print("See: https://www.alibabacloud.com/help/model-studio/developer-reference/error-code")


## safe saving method

save_interaction method saves each response record seperately so that our logs stay safe if the system is interrupted

## Prompts

The prompts stay the same for every skill. Only the skill input changes.

### Grouping

These are the initially provided questions regarding the skills:

1.	Explain the concept of [AGRI_SKILL] to a beginner with no prior background in sustainability.
2.	Provide a practical example of how [AGRI_SKILL] can be applied in a real-world organizational or industrial context.
3.	How does [AGRI_SKILL] contribute to environmental sustainability and long-term resource efficiency?
4.	Suggest a step-by-step learning path for someone who wants to acquire skills related to [AGRI_SKILL].
5.	Explain [AGRI_SKILL] for a university student in a technical field (e.g., engineering or computer science).


I decided to group them as follows:

|Prompt|Category|Questions|Notes|
|--------|---------|--------|-----------------------------|
|1|Foundational and Pedagogical|1, 4|Questions 1 and 4 focus on basic knowledge and explanation|
|2|Operations and Impact|2, 3|These questions are more practical, they focus on applications and impact|
|3|Technical Aspect - Specialisation|5|Question 5 focuses on technical depth and terminology|


In [ ]:
def create_prompt_1(input_value):
    system_text = """
    You are a career consultant specialized in agricultural skills.
    You provide information and tips concerning agricultural skills based on trustworthy data."""
    user_text = f"""
            ### INPUT DATA
            1. Skill name: {input_value["skill_name"]}
            2. Official ESCO skill description: {input_value["skill_description"]}
            3. Skill's ancestors for categorisation: {input_value["skill_ancestors"]}

            ### INSTRUCTIONS
            1. Explain the concept of the skill in a clear and simple way for a beginner. Use 3-4 sentences.
            2. Ensure that the explanation strictly reflects the provided skill description.
            3. Do not introduce information that is not supported by the provided description or ancestors.
            4. Provide a structured learning path for the skill consisting of 3-5 concrete and actionable steps.
            5. Each step must directly relate to the skill and progressively build competence.
            6. Each step should:
                - contain a short title (max 6 words)
                - contain a one sentence descripton
                - be specific, actionable, and directly related to the skill
                - build progressively from basic to advanced competence


            ### OUTPUT FORMAT
            Return a JSON object only.
            The answer should have the following form:
            {{
            "skill_name": "...",
            "skill_concept": "...",
            "learning_path": [
                {{
                  "step_title": "...",
                  "step_description": "..."
                }},
                {{
                  "step_title": "...",
                  "step_description": "..."
                }}
            ]
            }}
             """

    return user_text, system_text

def create_prompt_2(input_value):
    system_text = """
    You are an expert in agricultural strategies and sustainability.
    You provide scientifically accurate information concerning the application and environmental impact of agricultural skills."""
    user_text = f"""
      ### INPUT DATA
    1. Skill name: {input_value["skill_name"]}
    2. Official ESCO skill description: {input_value["skill_description"]}
    3. Skill's ancestors for categorisation: {input_value["skill_ancestors"]}

    ### INSTRUCTIONS
    1. Explain how the skill can be applied in an organization or an industry. Provide 3-5 application cases.
    2. Each case should:
      - contain a short title (max 6 words)
      - contain a one sentence descripton
      - be specific and directly related to the skill
    3. Explain how the skill relates to sustainability across the following dimensions:
      - environmental impact
      - resource efficiency
      - long-term effects
    4. Provide 2-3 sentences for each dimension.
    5. Ensure that the responses strictly reflect the provided skill description and its ancestors.
    6. Do not introduce information that is not supported by the provided description or ancestors. Do not include unrelated sustainability concepts.
    7. Use terminology consistent with professional agricultural and sustainability standards.


    ### OUTPUT FORMAT
    Return a JSON object only.
    The answer should have the following form:
    {{
      "skill_name": "...",
      "skill_applications": [
                {{
                  "application_title": "...",
                  "application_description": "..."
                }},
                {{
                  "application_title": "...",
                  "application_description": "..."
                }}
            ],
      "sustainability_connection": {{
        "environmental_impact": "...",
        "resource_efficiency": "...",
        "long_term_effects": "..."
      }}
    }}

              """
    return user_text, system_text

def create_prompt_3(input_value):
    system_text = """
    You are an expert in agricultural skills with strong technical knowledge.
    You provide precise and structured explanations aligned with ESCO definitions."""
    user_text = f"""
    ### INPUT DATA
    1. Skill name: {input_value["skill_name"]}
    2. Official ESCO skill description: {input_value["skill_description"]}
    3. Skill's ancestors for categorisation: {input_value["skill_ancestors"]}

    ### TASKS
    1. Provide a structured technical explanation of the skill for a university-level audience with a technical background.
    2. Structure the explanation as follows:
      - concise definition of the skill in 3-4 sentences
      - 3-5 key processes, methods, or components involved. Each item should be one sentence.
      - 3-5 practical or technical implications derived from the processes. Each item should be one sentence.
    3. Use precise and domain-relevant terminology.
    4. Ensure the explanation strictly reflects the provided skill description and ancestors.
    5. Do not introduce information that is not supported by the input.

    ### OUTPUT FORMAT
        Return a JSON object only.
        The answer should have the following form:
    {{
        "skill_name": "...",
        "technical_explanation": {{
          "definition": "...",
          "processes": [
            "...",
            "..."
          ],
          "technical_implications": [
            "...",
            "..."
          ]
        }}
    }}

    """

    return user_text, system_text


In [ ]:
import json

# function to seperately save each record for prompt
def save_interaction(record, prompt_type, model_type):
    with open(f"prompt_{prompt_type}_model_{model_type}.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

In [ ]:
import time
from datetime import datetime

input_data_sample = input_data[:5]
def run_multiple_prompts(input_data, prompt_type, model_type):
    """
    Runs multiple prompts on a specific model based on the input data.

    Args:
        input_data (list): List of input data for each prompt.
        prompt_type (str): Type of prompt to run. There are 3 types of prompts: based on the given type, a different function is called to create the prompt.
        model_type (str): Type of the model to use for generating responses.

    Returns:
        list: List of generated responses.
    """
    records = []
    for i, input_value in enumerate(input_data):
        if prompt_type == "1": user_text, system_text = create_prompt_1(input_value)
        elif prompt_type == "2": user_text, system_text = create_prompt_2(input_value)
        elif prompt_type == "3": user_text, system_text = create_prompt_3(input_value)
        else: raise ValueError("Invalid prompt type")

        if model_type == "1": record = generate_gemini_response(user_text, system_text)
        elif model_type == "2": record = generate_mistral_response(user_text, system_text)
        elif model_type == "3": record = generate_qwen_response(user_text, system_text)
        else: raise ValueError("Invalid model")

        # stores the response with all the necassary metadata
        records.append(record)
        save_interaction(record, prompt_type=prompt_type, model_type=model_type)
        print(f"Response {i + 1} stored \n")
        print(record["output"]["finish_reason"])
        print("\n")
        # adding pause to avoid requests per minute error
        time.sleep(2)

    return records



In [ ]:
records = run_multiple_prompts(input_data, "3", "3" )

In [ ]:
# check if the json objects are valid
df = pd.read_json('/content/prompt_2_model_2.jsonl', lines=True)

#df.head()

import json

def validate_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            try:
                json.loads(line)
            except json.JSONDecodeError as e:
                print(f"Error in line {i}: {e}")
                return False
    print("JSONL is valid!")
    return True

validate_jsonl("prompt_2_model_2.jsonl")